# 📌 정적 사이트 크롤링 완전 정복
## — 어떤 사이트든 스스로 크롤링하는 법

> **이 노트북의 핵심 목표**
> 특정 사이트 코드를 외우는 게 아니라,
> **어떤 새 사이트든 스스로 크롤링할 수 있는 능력**을 기른다.

---

## 이 노트북에서 다룰 내용

| 파트 | 내용 | 핵심 기술 |
|------|------|----------|
| 0 | 크롤링 전 30초 체크리스트 | 정적/동적 판별 + robots.txt |
| 0-3 | **개발자도구 선택자 찾기 완전 가이드** | DevTools 마스터 ← 가장 중요! |
| 1 | 사람인 채용공고 (심화) | 여러 키워드 비교, 기술스택 추출 |
| 2 | 알라딘 베스트셀러 | 새 사이트 탐색 방법론 실습 |
| 3 | 범용 크롤링 템플릿 | 어떤 사이트든 5분 안에 시작 |
| 4 | 연습문제 | 잡코리아, 직접 찾은 사이트 |

---
## Part 0. 크롤링 전 30초 체크리스트

### 0-1. 정적/동적 판별 (Ctrl+U 테스트)

새 사이트를 크롤링하기 전에 **반드시** 확인해야 하는 것:

```
방법: 크롬에서 Ctrl+U → 소스 탭 → Ctrl+F → 수집하고 싶은 텍스트 검색

  텍스트가 보이면 → 정적 사이트 → BeautifulSoup OK ✅
  텍스트가 없으면 → 동적 사이트 → Selenium 필요 🤖
```

### 정적 사이트 예시 (이 노트북에서 다루는 것들)

| 사이트 | 확인 방법 | 결과 |
|--------|-----------|------|
| 사람인 | Ctrl+U → 공고 제목 검색 | ✅ 정적 |
| 알라딘 | Ctrl+U → 책 제목 검색 | ✅ 정적 |
| 잡코리아 | Ctrl+U → 공고 제목 검색 | ✅ 정적 |
| 무신사 | Ctrl+U → 상품명 검색 | ❌ 동적 (Selenium 필요) |

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# ─── 표준 헤더 — 앞으로 항상 이걸 사용 ────────────────────────
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def is_static(url, test_text):
    """
    사이트가 정적인지 코드로 확인합니다.
    test_text: 해당 사이트에서 수집하고 싶은 텍스트 중 하나
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.encoding = 'utf-8'
        
        if test_text in response.text:
            print(f"✅ 정적 사이트!  '{test_text}'가 소스에 있음 → BeautifulSoup 가능")
        else:
            print(f"❌ 동적 가능성! '{test_text}'가 소스에 없음 → Selenium 검토 필요")
        
        print(f"   상태코드: {response.status_code} | 소스크기: {len(response.text):,} 글자")
        return response
    except Exception as e:
        print(f"오류: {e}")
        return None


# ─── 테스트 ────────────────────────────────────────────────────
print("=== 사람인 ===")
is_static('https://www.saramin.co.kr/zf_user/search/recruit?searchword=데이터분석', '데이터분석')

print()
print("=== 알라딘 ===")
is_static('https://www.aladin.co.kr/shop/common/wbest.aspx?BestType=Bestseller&BranchType=1&CID=0', '베스트셀러')

In [ ]:
# robots.txt 확인 — 크롤링 전 습관!

def check_robots(site_url):
    """사이트의 robots.txt를 확인합니다."""
    robots_url = site_url.rstrip('/') + '/robots.txt'
    try:
        r = requests.get(robots_url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            print(f"=== {site_url} robots.txt ===")
            print(r.text[:700])
        else:
            print(f"robots.txt 없음 (상태코드: {r.status_code})")
    except Exception as e:
        print(f"오류: {e}")

check_robots('https://www.saramin.co.kr')
print()
check_robots('https://www.aladin.co.kr')

---
### 0-2. 개발자도구로 선택자 찾기 — 완전 가이드 ★★★

이것이 이 노트북에서 **가장 중요한 내용**입니다.
이걸 익히면 어떤 새로운 사이트든 **스스로** 크롤링할 수 있어요.

---

#### 방법 1: Copy Selector (가장 빠름)

```
1. 크롬에서 수집하고 싶은 요소 위에 마우스 올리기
2. 우클릭 → "검사" (Inspect)
3. 파란색으로 강조된 HTML 태그에서 우클릭
4. Copy → Copy selector
5. soup.select_one('여기에 붙여넣기') 로 사용
```

> ⚠️ Copy Selector 결과 예: `#container > ul > li:nth-child(1) > div.info > h3.title`
> 이렇게 길면 사이트 구조가 조금만 바뀌어도 에러납니다.
> 가능하면 **의미 있는 class 기반의 짧은 선택자**를 쓰세요.

---

#### 방법 2: Inspector에서 직접 읽기 (더 안정적)

```
1. F12 (또는 Ctrl+Shift+I)
2. 좌측 상단 Inspector 포인터 아이콘 클릭 (사각형+마우스 아이콘)
3. 원하는 텍스트 클릭
4. Elements 탭에 강조된 HTML 확인

예시: <li class="product-item active">  →  soup.select('li.product-item')
      <div id="search-result">          →  soup.select_one('#search-result')
      <a href="/jobs/1">               →  soup.select('a[href*="/jobs/"]')
```

---

#### 핵심: 반복 패턴 찾기

목록 데이터를 수집할 때 가장 중요한 것:
**"여러 개가 반복되는 단위(카드/아이템 하나)"를 먼저 찾는다.**

```html
<!-- 이런 구조에서 -->
<ul class="product-list">           ← 전체 컨테이너 (한 번만 있음)
  <li class="product-item">         ← 반복 단위! ← 이걸 먼저 잡자
    <h3 class="name">립밤</h3>
    <p class="price">12,000원</p>
  </li>
  <li class="product-item">         ← 두 번째 반복
    <h3 class="name">선크림</h3>
    <p class="price">25,000원</p>
  </li>
</ul>
```

```python
# 3단계 수집 패턴 (이것만 기억!)

# Step 1: 반복 단위 전부 찾기
items = soup.select('li.product-item')

# Step 2 + 3: 반복문으로 각 아이템에서 데이터 추출
for item in items:
    name  = item.select_one('.name').text.strip()
    price = item.select_one('.price').text.strip()
```

---

#### CSS 선택자 치트시트

| 선택자 | 의미 | 예시 |
|--------|------|------|
| `div` | div 태그 | `soup.select('div')` |
| `.name` | class="name" | `soup.select('.name')` |
| `#header` | id="header" | `soup.select_one('#header')` |
| `div.card` | div이면서 class=card | `soup.select('div.card')` |
| `div > p` | div 바로 아래 p | `soup.select('div > p')` |
| `div p` | div 안의 모든 p | `soup.select('div p')` |
| `a[href]` | href 속성 있는 a | `soup.select('a[href]')` |
| `a[href*="jobs"]` | href에 'jobs' 포함 | `soup.select('a[href*="jobs"]')` |
| `li.item.active` | item이면서 active 클래스 | `soup.select('li.item.active')` |

---
## Part 1. 채용공고 — 사람인 (심화)

### 사이트 정보
- **URL 패턴**: `https://www.saramin.co.kr/zf_user/search/recruit?searchword=키워드&recruitPage=N`
- **정적 확인**: ✅ Ctrl+U 소스에 공고 데이터 있음

### HTML 구조 (개발자도구로 확인한 것)
```html
<div class="item_recruit">           ← 공고 하나 (반복 단위)
  <strong class="corp_name">
    <a>회사명</a>                     ← 회사명
  </strong>
  <h2 class="job_tit">
    <a>공고 제목</a>                   ← 공고 제목
  </h2>
  <div class="job_condition">
    <span>지역</span>                  ← 지역, 경력, 학력, 고용형태
    <span>경력</span>
    <span>학력</span>
    <span>고용형태</span>
  </div>
  <div class="job_date">
    <span class="date">마감일</span>  ← 마감일
  </div>
</div>
```

### 이번 목표: 여러 키워드를 한번에 수집해서 직무별로 비교하기

In [ ]:
def crawl_saramin(keyword, max_page=3):
    """사람인에서 키워드 채용공고를 수집합니다."""
    all_jobs = []
    
    for page in range(1, max_page + 1):
        url = (
            f'https://www.saramin.co.kr/zf_user/search/recruit'
            f'?searchType=search&searchword={keyword}&recruitPage={page}'
        )
        
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.encoding = 'utf-8'
            
            if response.status_code != 200:
                print(f"  [{keyword}] {page}p 오류: {response.status_code}")
                break
            
            soup = BeautifulSoup(response.text, 'html.parser')
            jobs = soup.select('div.item_recruit')
            
            if not jobs:
                print(f"  [{keyword}] {page}p 공고 없음, 종료")
                break
            
            for job in jobs:
                company_el  = job.select_one('strong.corp_name a')
                title_el    = job.select_one('h2.job_tit a')
                conds       = [c.text.strip() for c in job.select('div.job_condition span')]
                deadline_el = job.select_one('div.job_date span.date')
                
                all_jobs.append({
                    '키워드':   keyword,
                    '페이지':   page,
                    '회사명':   company_el.text.strip() if company_el else None,
                    '공고제목': title_el.text.strip()   if title_el   else None,
                    '지역':     conds[0] if len(conds) > 0 else None,
                    '경력':     conds[1] if len(conds) > 1 else None,
                    '학력':     conds[2] if len(conds) > 2 else None,
                    '고용형태': conds[3] if len(conds) > 3 else None,
                    '마감일':   deadline_el.text.strip() if deadline_el else None,
                })
            
            print(f"  [{keyword}] {page}p → {len(jobs)}건")
            time.sleep(1)
        
        except Exception as e:
            print(f"  [{keyword}] {page}p 예외: {e}")
            break
    
    return all_jobs

print("crawl_saramin 함수 정의 완료!")

In [ ]:
# 여러 직무 키워드 한번에 수집
# 시간이 걸립니다 (키워드당 약 3초 × 3페이지)

keywords = ['데이터분석', '파이썬개발자', 'SQL', '머신러닝']
all_jobs = []

for kw in keywords:
    print(f"\n[{kw}] 수집 중...")
    jobs = crawl_saramin(kw, max_page=3)
    all_jobs.extend(jobs)
    time.sleep(2)  # 키워드 사이 추가 대기

df_jobs = pd.DataFrame(all_jobs)
print(f"\n총 {len(df_jobs)}건 수집 완료!")
print()
print(df_jobs.groupby('키워드').size().rename('공고수').to_string())

In [ ]:
# ─── 직무별 비교 분석 ───────────────────────────────────────────

print("=== 직무별 경력 조건 분포 ===")
pivot_exp = df_jobs.groupby(['키워드', '경력']).size().unstack(fill_value=0)
entry_cols = [c for c in pivot_exp.columns if c in ['경력무관', '신입', '신입·경력']]
if entry_cols:
    print(pivot_exp[entry_cols].to_string())
else:
    print("신입 관련 컬럼을 찾지 못했습니다. 경력 컬럼 전체:")
    print(pivot_exp.head())

print()
print("=== 지역별 공고 수 (상위 8) ===")
print(df_jobs['지역'].value_counts().head(8).to_string())

print()
print("=== 공고 제목에서 기술 스택 추출 ===")
# 제목에 언급된 기술 키워드 집계
tech_keywords = [
    'Python', 'SQL', 'R', 'Tableau', 'Power BI', 'Excel',
    '머신러닝', '딥러닝', 'AI', 'AWS', 'Azure', 'Spark', 'hadoop'
]

tech_counts = {}
for tech in tech_keywords:
    count = df_jobs['공고제목'].str.contains(tech, case=False, na=False).sum()
    if count > 0:
        tech_counts[tech] = count

tech_counts_sorted = sorted(tech_counts.items(), key=lambda x: x[1], reverse=True)
for tech, count in tech_counts_sorted:
    bar = '█' * count
    print(f"  {tech:12}: {bar} ({count}건)")

In [ ]:
# ─── 실전 활용: 신입/경력무관 중 서울 공고 필터링 ────────────────

entry_keywords = ['경력무관', '신입', '신입·경력']

entry_seoul = df_jobs[
    df_jobs['경력'].isin(entry_keywords) &
    df_jobs['지역'].str.contains('서울', na=False)
].copy()

print(f"신입/경력무관 + 서울 공고: {len(entry_seoul)}건")
print()
print(entry_seoul[['키워드', '회사명', '공고제목', '경력', '지역']]
      .head(15)
      .to_string(index=False))

---
## Part 2. 도서 — 알라딘 베스트셀러
### 새 사이트 탐색 방법론 실습

이 섹션에서는 **처음 보는 사이트를 탐색하는 과정**을 그대로 보여드립니다.
코드를 외우는 게 아니라, 탐색 → 파악 → 수집의 흐름을 익히세요.

**알라딘 베스트셀러 URL:**
```
https://www.aladin.co.kr/shop/common/wbest.aspx?BestType=Bestseller&BranchType=1&CID=0
```

### 탐색 순서
```
Step 1: 정적 사이트인지 확인 (Ctrl+U 또는 is_static 함수)
Step 2: HTML 구조 출력해서 반복 패턴 찾기
Step 3: F12로 선택자 확인
Step 4: 수집 코드 작성
Step 5: 데이터 정제 + 분석
```

In [ ]:
# Step 1: 알라딘 페이지 가져오기 + 정적 확인

aladin_url = 'https://www.aladin.co.kr/shop/common/wbest.aspx?BestType=Bestseller&BranchType=1&CID=0'

response_aladin = requests.get(aladin_url, headers=HEADERS, timeout=15)
response_aladin.encoding = 'utf-8'

print(f"상태 코드: {response_aladin.status_code}")
print(f"소스 크기: {len(response_aladin.text):,} 글자")
print()

# 정적 확인: '위', '베스트' 텍스트가 소스에 있는지
if '베스트셀러' in response_aladin.text:
    print("✅ '베스트셀러' 소스에 있음 → 정적 사이트!")
else:
    print("텍스트 없음. 다른 텍스트로 시도해보세요.")

In [ ]:
# Step 2: HTML 구조 탐색 — 반복 패턴 찾기

soup_aladin = BeautifulSoup(response_aladin.text, 'html.parser')

# ─── 탐색 도구 1: 반복될 것 같은 클래스 찾기 ─────────────────
print("=== 'book', 'list', 'item', 'best' 포함 클래스 찾기 ===")
seen = set()
for tag in soup_aladin.find_all(class_=True):
    classes = tag.get('class', [])
    for cls in classes:
        if re.search(r'book|list|item|best|rank', cls, re.I) and cls not in seen:
            seen.add(cls)
            print(f"  <{tag.name}> class='{cls}' | 텍스트: {tag.get_text(strip=True)[:40]}")

print()
print("─" * 50)
print("위 결과에서 반복되는 책 단위 클래스를 찾아보세요.")
print("찾은 후 다음 셀의 BOOK_ITEM_SELECTOR 를 수정하세요.")

In [ ]:
# Step 2 (계속): 특정 클래스로 태그 수 확인
# — 위 탐색 결과를 보고 후보 클래스를 여기서 시험해보세요

def count_tags(soup, selector, preview_chars=60):
    """특정 선택자로 몇 개의 태그가 있는지 확인하고 미리보기를 보여줍니다."""
    tags = soup.select(selector)
    print(f"'{selector}': {len(tags)}개 발견")
    for i, tag in enumerate(tags[:3], 1):
        text = tag.get_text(strip=True)
        print(f"  [{i}] {text[:preview_chars]}")
    return tags

# 후보 선택자들을 시험해보세요
# 탐색 셀에서 찾은 클래스를 아래에 입력하세요

print("=== 후보 선택자 테스트 ===")
print("(위 탐색 셀 결과를 보고 아래에 실제 클래스를 입력하세요)")
print()

# 예시 (실제 클래스로 교체해야 합니다)
candidates = [
    'div.ss_book_list',   # 후보 1
    'li',                 # 후보 2 (모든 li)
    'table tr',           # 후보 3 (테이블 구조인 경우)
]

for selector in candidates:
    tags = soup_aladin.select(selector)
    print(f"'{selector}': {len(tags)}개")

### Step 3: F12로 정확한 선택자 확인하기

```
1. 크롬에서 알라딘 베스트셀러 페이지 열기:
   https://www.aladin.co.kr/shop/common/wbest.aspx?BestType=Bestseller&BranchType=1&CID=0

2. F12 → Inspector 포인터 아이콘 클릭 (좌측 상단)

3. 책 제목 텍스트 클릭
   → Elements 탭에서 파란색 강조된 HTML 확인

4. 확인해야 할 것들:
   □ 책 하나의 반복 단위 태그/클래스는? (예: li.ss_book_list)
   □ 제목은 어떤 태그/클래스? (예: b.bo3)
   □ 저자/출판사는? 
   □ 가격은?
   □ 순위는 어디에 있나? (텍스트로 있나? data 속성으로 있나?)

5. 아래 셀의 선택자 변수들을 확인한 값으로 채우세요.
```

> 💡 **Tip**: Elements 탭에서 `Ctrl+F`를 누르면 HTML 소스 안에서 텍스트 검색을 할 수 있어요!

In [ ]:
# Step 4: 수집 코드
# ─────────────────────────────────────────────────────────────
# 아래 선택자들을 F12로 확인한 값으로 교체하세요.
# 탐색 셀의 출력을 보면 힌트가 있어요!
# ─────────────────────────────────────────────────────────────

ITEM_SEL   = 'div.ss_book_list'   # 책 하나의 반복 단위  ← F12로 확인 후 수정
TITLE_SEL  = 'b.bo3'              # 제목                ← F12로 확인 후 수정
AUTHOR_SEL = '.ss_author'         # 저자/출판사          ← F12로 확인 후 수정  
PRICE_SEL  = '.ss_p2'             # 할인가              ← F12로 확인 후 수정

# ─────────────────────────────────────────────────────────────

items = soup_aladin.select(ITEM_SEL)
print(f"발견된 아이템 수: {len(items)}")

if not items:
    print()
    print("⚠️  아이템을 찾지 못했습니다.")
    print("   탐색 셀의 출력을 보고 ITEM_SEL 값을 수정해보세요.")
    print("   또는 F12로 직접 확인하세요.")
else:
    rows_aladin = []
    for item in items:
        title_el  = item.select_one(TITLE_SEL)
        author_el = item.select_one(AUTHOR_SEL)
        price_el  = item.select_one(PRICE_SEL)
        
        rows_aladin.append({
            '제목':   title_el.text.strip()  if title_el  else None,
            '저자':   author_el.text.strip() if author_el else None,
            '가격':   price_el.text.strip()  if price_el  else None,
        })
    
    df_aladin = pd.DataFrame(rows_aladin)
    print(df_aladin.to_string(index=False))

In [ ]:
# Step 5: 수집된 데이터 분석 (df_aladin이 채워진 후 실행)

# 가격 숫자 변환
def price_to_int(price_str):
    """'27,000원' → 27000"""
    if not price_str:
        return None
    nums = re.sub(r'[^0-9]', '', price_str)
    return int(nums) if nums else None

if 'df_aladin' in dir() and len(df_aladin) > 0:
    df_aladin['가격_숫자'] = df_aladin['가격'].apply(price_to_int)
    
    print("=== 알라딘 베스트셀러 분석 ===")
    print(f"수집된 책 수: {len(df_aladin)}권")
    print(f"평균 가격:   {df_aladin['가격_숫자'].mean():,.0f}원")
    print(f"최저 가격:   {df_aladin['가격_숫자'].min():,.0f}원")
    print(f"최고 가격:   {df_aladin['가격_숫자'].max():,.0f}원")
    print()
    print(df_aladin[['제목', '저자', '가격']].head(10).to_string(index=False))
else:
    print("df_aladin 이 비어있습니다. 선택자를 수정하고 수집 셀을 먼저 실행하세요.")

---
## Part 3. 범용 크롤링 탐색 도구
### 어떤 사이트든 5분 안에 구조 파악하기

Part 2에서 배운 탐색 방법론을 함수로 만들었습니다.
앞으로 새 사이트를 만날 때마다 이 함수들을 먼저 쓰세요.

In [ ]:
def explore_new_site(url, search_text=None):
    """
    새 사이트의 구조를 탐색합니다.
    이 함수 하나로 수집 가능 여부와 HTML 구조를 빠르게 파악할 수 있어요.
    
    url:         탐색할 사이트 URL
    search_text: 소스에서 찾을 텍스트 (정적 확인용)
    """
    print(f"[탐색 시작] {url}")
    print("=" * 60)
    
    # 1. 페이지 가져오기
    response = requests.get(url, headers=HEADERS, timeout=15)
    response.encoding = 'utf-8'
    print(f"상태 코드: {response.status_code} | 소스크기: {len(response.text):,} 글자")
    
    # 2. 정적 확인
    if search_text:
        if search_text in response.text:
            print(f"✅ 정적 사이트 → '{search_text}' 소스에 있음")
        else:
            print(f"❌ 동적 가능성 → '{search_text}' 소스에 없음 (Selenium 검토)")
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 3. 반복 가능한 클래스 찾기
    print()
    print("─── 반복 패턴 후보 (list/item/card/result 관련) ───")
    class_counter = {}
    for tag in soup.find_all(class_=True):
        for cls in tag.get('class', []):
            class_counter[cls] = class_counter.get(cls, 0) + 1
    
    # 5번 이상 반복 + 의미있는 이름인 클래스 출력
    for cls, count in sorted(class_counter.items(), key=lambda x: -x[1]):
        if count >= 5 and re.search(r'item|card|list|row|result|product|job|article', cls, re.I):
            sample = soup.select_one(f'.{cls}')
            preview = sample.get_text(strip=True)[:40] if sample else ''
            print(f"  .{cls}: {count}회 반복 | 예시: {preview}")
    
    # 4. 소스 미리보기
    print()
    print("─── HTML 소스 앞부분 (1500자) ───")
    print(soup.prettify()[:1500])
    
    return soup


# 사용 예시
print("explore_new_site 함수 정의 완료!")
print()
print("사용법:")
print("  soup = explore_new_site('사이트URL', '찾을텍스트')")
print("  → 구조 탐색 결과 출력 + soup 반환")

In [ ]:
# ─── 범용 수집 템플릿 ────────────────────────────────────────
# 새 사이트를 만나면 이 템플릿을 복사하고 선택자만 채워넣으세요.

def crawl_site(
    base_url,
    item_selector,     # 반복 단위 선택자 (예: 'li.product-item')
    fields,            # {컬럼명: 선택자} 딕셔너리
    page_param='page', # 페이지 파라미터 이름
    max_page=3
):
    """
    범용 크롤링 함수 — 선택자만 바꾸면 어떤 사이트든 수집
    
    사용 예시:
    df = crawl_site(
        base_url='https://example.com/list?q=키워드',
        item_selector='li.job-item',
        fields={
            '회사명': '.company',
            '제목':   'h3.title',
            '위치':   '.location',
        },
        page_param='page',
        max_page=3
    )
    """
    all_rows = []
    
    for page in range(1, max_page + 1):
        # 페이지 파라미터가 URL에 이미 있으면 &, 없으면 ?
        sep = '&' if '?' in base_url else '?'
        url = f"{base_url}{sep}{page_param}={page}"
        
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            resp.encoding = 'utf-8'
            soup = BeautifulSoup(resp.text, 'html.parser')
            items = soup.select(item_selector)
            
            if not items:
                print(f"  {page}p: 아이템 없음, 종료")
                break
            
            for item in items:
                row = {'페이지': page}
                for col_name, selector in fields.items():
                    el = item.select_one(selector)
                    row[col_name] = el.text.strip() if el else None
                all_rows.append(row)
            
            print(f"  {page}p: {len(items)}개")
            time.sleep(1)
        
        except Exception as e:
            print(f"  {page}p 오류: {e}")
            break
    
    return pd.DataFrame(all_rows)


print("crawl_site 함수 정의 완료!")

---
## Part 4. 연습문제

### 문제 1 (기초) — 잡코리아 채용공고

잡코리아에서 '데이터분석' 키워드 공고를 수집하세요.

```
URL: https://www.jobkorea.co.kr/Search/?stext=데이터분석&Page_No=1

단계:
1. Ctrl+U로 정적 확인
2. explore_new_site() 로 구조 탐색
3. F12로 선택자 확인
4. crawl_site() 또는 직접 함수로 수집
5. 사람인 데이터와 비교 분석
```

---

### 문제 2 (중급) — 두 채용 사이트 비교

문제 1에서 수집한 잡코리아 데이터와 Part 1의 사람인 데이터를 합쳐서
직무별/지역별로 비교 분석하세요.

```python
# 힌트: 출처 컬럼 추가 후 concat
df_saramin['출처'] = '사람인'
df_jobkorea['출처'] = '잡코리아'
df_combined = pd.concat([df_saramin, df_jobkorea], ignore_index=True)
```

---

### 문제 3 (고급) — 나만의 사이트 크롤링

관심 있는 사이트를 하나 골라서 스스로 크롤링해보세요.

```
추천 사이트:
  채용공고: 인크루트 (incruit.com), 원티드 (wanted.co.kr, 일부 정적)
  도서:     교보문고 (kyobobook.co.kr)
  연습용:   books.toscrape.com, quotes.toscrape.com

체크리스트:
  □ Ctrl+U로 정적 확인
  □ robots.txt 확인
  □ explore_new_site()로 구조 탐색
  □ F12로 선택자 확인
  □ 수집 후 CSV 저장
```

In [ ]:
# 연습문제 1 힌트: 잡코리아 탐색

# 주석 해제 후 실행해보세요
# soup_jk = explore_new_site(
#     'https://www.jobkorea.co.kr/Search/?stext=데이터분석&Page_No=1',
#     '데이터분석'
# )

# 탐색 결과를 보고 아래 선택자를 채워넣으세요
# df_jk = crawl_site(
#     base_url='https://www.jobkorea.co.kr/Search/?stext=데이터분석',
#     item_selector='TODO: F12로 확인한 공고 반복 단위 선택자',
#     fields={
#         '회사명':   'TODO',
#         '공고제목': 'TODO',
#         '지역':     'TODO',
#         '경력':     'TODO',
#     },
#     page_param='Page_No',
#     max_page=3
# )

print("주석을 해제하고 실행해보세요!")

---
## Part 5. 전체 데이터 저장

In [ ]:
import os

save_dir = os.path.join('data', 'results')
os.makedirs(save_dir, exist_ok=True)

# 사람인 채용공고 저장
if 'df_jobs' in dir() and len(df_jobs) > 0:
    path = os.path.join(save_dir, '사람인_채용공고_심화.csv')
    df_jobs.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"✅ 사람인: {len(df_jobs)}행 → {path}")

# 알라딘 저장 (수집된 경우)
if 'df_aladin' in dir() and len(df_aladin) > 0:
    path = os.path.join(save_dir, '알라딘_베스트셀러.csv')
    df_aladin.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"✅ 알라딘: {len(df_aladin)}행 → {path}")

print()
print("💡 encoding='utf-8-sig' → 엑셀에서 열어도 한글이 안 깨짐")

---
## ✅ 핵심 정리

### 새 사이트 크롤링 5단계 프로세스

```
Step 1: Ctrl+U 또는 is_static() 로 정적 확인
          → 없으면 Selenium 노트북으로!

Step 2: robots.txt 확인 (check_robots)

Step 3: explore_new_site() 로 구조 탐색
          → 반복 패턴 후보 확인

Step 4: F12 → Inspector로 정확한 선택자 찾기
          → 반복 단위 선택자, 데이터 선택자

Step 5: crawl_site() 또는 직접 함수로 수집
          → time.sleep(1) 항상!
          → encoding='utf-8-sig' 로 CSV 저장
```

---

### BeautifulSoup 핵심 패턴 (한눈에)

```python
# 탐색
soup.select('선택자')           # 전부 (리스트)
soup.select_one('선택자')       # 하나 (또는 None)

# 데이터 추출
el.text.strip()                 # 텍스트
el.get('href')                  # 속성값 (없으면 None)
el.get('class', [])             # 클래스 목록

# None 안전 처리
el.text.strip() if el else None

# 수집 구조
for item in soup.select('li.item'):
    name  = item.select_one('.name').text.strip() if item.select_one('.name') else None
    price = item.select_one('.price').text.strip() if item.select_one('.price') else None
    rows.append({'이름': name, '가격': price})

df = pd.DataFrame(rows)
df.to_csv('결과.csv', index=False, encoding='utf-8-sig')
```

---

> **다음 노트북**: Selenium × AI 프롬프트 활용법
> → 코드를 외우지 말고, AI에게 생성시키고 실행하는 법